## PLNet Prediction Visualization on Oxparis
In this notebook, we visualize the point and line predictions of PLNet on the Oxparis (MiniDepth) dataset.

### Imports

In [1]:
import torch
from matplotlib import pyplot as plt
from gluefactory.datasets import get_dataset
from gluefactory.models import get_model

### Define utility functions

In [2]:
def plot_images(images, titles=None, cmaps='gray', figsize=(20, 10)):
    n = len(images)
    fig, ax = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        ax = [ax]
    for i in range(n):
        ax[i].imshow(images[i], cmap=cmaps[i] if isinstance(cmaps, list) else cmaps)
        if titles:
            ax[i].set_title(titles[i])
        ax[i].axis('off')
    plt.show()

def plot_lines(lines, line_scores=None, ax=None, color='r', linewidth=1, alpha=1.0):
    if ax is None:
        ax = plt.gca()
    for i, line in enumerate(lines):
        if line_scores is not None:
            a = alpha * line_scores[i]
        else:
            a = alpha
        ax.plot([line[0, 0], line[1, 0]], [line[0, 1], line[1, 1]], color=color, linewidth=linewidth, alpha=a)

def get_model_size_on_gpu(model):
    param_size = 0
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()
    buffer_size = 0
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()

    size_all_mb = (param_size + buffer_size) / 1024**2
    print(f'Model size: {size_all_mb:.3f} MB')
    
    if torch.cuda.is_available():
        # This is just a rough estimate of memory used by the model on GPU
        print(f'Active GPU memory: {torch.cuda.memory_allocated() / 1024**2:.3f} MB')
        print(f'Max GPU memory: {torch.cuda.max_memory_allocated() / 1024**2:.3f} MB')


### Load the Dataset

In [3]:
dset_conf = {
    "name": "minidepth",
    "train_batch_size": 1,
    "val_batch_size": 1,
    "test_batch_size": 1,
    "reshape": 800,
    "load_features": {
        "do": False,
        "check_exists": True,
        "point_gt": {
            "data_keys": ["gt_keypoints", "gt_keypoints_scores"],
            "use_superpoint_kp_gt": True,
        },
        "line_gt": {
            "data_keys": ["deeplsd_distance_field", "deeplsd_angle_field"],
        },
    },
}
dataset = get_dataset("minidepth")(dset_conf)
loader = dataset.get_dataset(split="val")
print(f"Dataset size: {len(loader)}")


/Users/rkre/miniconda3/envs/jpl_gluefactory/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[01/25/2026 15:45:59 gluefactory.datasets.base_dataset INFO] Creating dataset MiniDepthDataset
/Users/rkre/PycharmProjects/glue-factory-new/gluefactory/datasets/augmentations.py:130: UserWarning: Argument(s) 'always_apply' are not valid for transform FromFloat
  self.preprocess = A.FromFloat(always_apply=True, dtype="uint8")
/Users/rkre/PycharmProjects/glue-factory-new/gluefactory/datasets/augmentations.py:131: UserWarning: Argument(s) 'always_apply' are not valid for transform ToFloat
  self.postprocess = A.ToFloat(always_apply=True)
[01/25/2026 15:45:59 gluefactory.datasets.minidepth INFO] NUMBER OF IMAGES: 850


Dataset size: 850


In [4]:
import gluefactory.models.extractors.pl_net_extractor

Successfully imported PLNet components from  /Users/rkre/PycharmProjects/glue-factory-new/other/PLNet


### Initialize PLNet

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plnet_conf = {
    "name": "extractors.pl_net_extractor",
    "max_num_junctions": 512,
    "max_num_lines": 512,
    "junction_threshold": 0.008,
    "line_threshold": 0.05,
}
model = get_model("extractors.pl_net_extractor")(plnet_conf).to(device)
model.eval()

print("Model information:")
get_model_size_on_gpu(model)


Initialize PLNet....


--2026-01-25 15:47:03--  https://entuedu-my.sharepoint.com/:u:/g/personal/kuan_xu_staff_main_ntu_edu_sg/EbQy7pSPVNFDrP81aloP-O8BA3W0HlOqFsTi6p20KGH9xA?e=mFgVdU&download=1
Resolving entuedu-my.sharepoint.com (entuedu-my.sharepoint.com)... 2620:1ec:8fa::10, 2620:1ec:8f8::10, 13.107.138.10, ...
Connecting to entuedu-my.sharepoint.com (entuedu-my.sharepoint.com)|2620:1ec:8fa::10|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: /personal/kuan_xu_staff_main_ntu_edu_sg/Documents/AirVIO/PLNet/model/plnet.pth?ga=1 [following]
--2026-01-25 15:47:04--  https://entuedu-my.sharepoint.com/personal/kuan_xu_staff_main_ntu_edu_sg/Documents/AirVIO/PLNet/model/plnet.pth?ga=1
Reusing existing connection to [entuedu-my.sharepoint.com]:443.
HTTP request sent, awaiting response... 200 OK
Length: 116534793 (111M) [application/octet-stream]
Saving to: ‘/Users/rkre/PycharmProjects/glue-factory-new/data/weights/plnet.pth’

     0K .......... .......... .......... .......... ........

Loaded SuperPoint model


..... 95%  832K 0s
109200K .......... .......... .......... .......... .......... 95%  197M 0s
109250K .......... .......... .......... .......... .......... 96%  136M 0s
109300K .......... .......... .......... .......... .......... 96% 48.9M 0s
109350K .......... .......... .......... .......... .......... 96%  708M 0s
109400K .......... .......... .......... .......... .......... 96%  751M 0s
109450K .......... .......... .......... .......... .......... 96%  373M 0s
109500K .......... .......... .......... .......... .......... 96%  814M 0s
109550K .......... .......... .......... .......... .......... 96% 56.4M 0s
109600K .......... .......... .......... .......... .......... 96%  788M 0s
109650K .......... .......... .......... .......... .......... 96%  678M 0s
109700K .......... .......... .......... .......... .......... 96%  814M 0s
109750K .......... .......... .......... .......... .......... 96%  339M 0s
109800K .......... .......... .......... .......... .......... 96%  7

AssertionError: Torch not compiled with CUDA enabled

### Run Inference and Visualize

In [ ]:
def visualize_predictions(data, model, device):
    img = data["image"].to(device).unsqueeze(0)
    with torch.no_grad():
        pred = model({"image": img})
    
    # Move to CPU for visualization
    img_cpu = img[0].cpu().permute(1, 2, 0).numpy()
    if img_cpu.shape[2] == 1:
        img_cpu = img_cpu.squeeze(2)
    
    kpts = pred["keypoints"][0].cpu().numpy()
    lines = pred["lines"][0].cpu().numpy()
    
    fig, ax = plt.subplots(1, 2, figsize=(20, 10))
    
    # Plot junctions
    ax[0].imshow(img_cpu, cmap='gray')
    ax[0].scatter(kpts[:, 0], kpts[:, 1], s=5, c='r', marker='o')
    ax[0].set_title(f"PLNet Junctions ({len(kpts)})")
    ax[0].axis('off')
    
    # Plot lines
    ax[1].imshow(img_cpu, cmap='gray')
    plot_lines(lines, ax=ax[1], color='r', linewidth=1)
    ax[1].set_title(f"PLNet Lines ({len(lines)})")
    ax[1].axis('off')
    
    plt.tight_layout()
    plt.show()


In [ ]:
# Visualize a few examples
for i in range(3):
    print(f"Example {i}")
    data = loader[i]
    visualize_predictions(data, model, device)
